# EDI free model feasibility
Private development diagnostic, 22 October 2026. Select GPU T4 x2 and enable internet for package/model download. This notebook runs two neutral requests on the pinned Qwen model, then stops its vLLM server. It does not run EDI or the Meadows benchmark. The temporary environment/model cache disappear with the VM. Preserve output reports before stopping the session. No OpenAI credential belongs in this notebook. The native sampler is intentional: the first FlashInfer sampler attempt failed and is retained in the repository evidence.


## Hardware inventory


In [ ]:
# Neutral host inventory only: no EDI benchmark, credentials, or model weights.
import json, platform, subprocess, importlib.util
from pathlib import Path
report = {"purpose":"EDI free GPU feasibility", "benchmark_started":False,
          "model_loaded":False, "paid_fallback":False, "python":platform.python_version()}
result = subprocess.run(
    ["nvidia-smi","--query-gpu=name,memory.total,driver_version","--format=csv,noheader"],
    text=True, capture_output=True, check=False)
report["nvidia_smi_returncode"] = result.returncode
report["gpus"] = result.stdout.strip().splitlines()
report["packages"] = {name: bool(importlib.util.find_spec(name)) for name in ["torch","vllm","transformers"]}
if report["packages"]["torch"]:
    import torch
    report["torch"] = torch.__version__
    report["cuda_available"] = torch.cuda.is_available()
    report["devices"] = [{"index":i, "name":torch.cuda.get_device_name(i),
                          "vram_bytes":torch.cuda.get_device_properties(i).total_memory,
                          "compute_capability":list(torch.cuda.get_device_capability(i))}
                         for i in range(torch.cuda.device_count())]
out = Path("/kaggle/working/edi-free-gpu-inventory.json")
out.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))
print("Inventory only. Do not treat this as a model-fit or isolation pass.")


## Isolated installation


In [ ]:
# Isolated free-session installation; no model weights or EDI benchmark yet.
import json, shutil, subprocess, sys, time, venv
from pathlib import Path
root=Path("/kaggle/temp/edi-free-vllm"); root.mkdir(parents=True,exist_ok=True)
free=shutil.disk_usage(root).free
print("Temporary disk free GiB:",round(free/2**30,1),flush=True)
assert free > 35*2**30, "Insufficient free temporary disk; stop without model download"
venv.EnvBuilder(with_pip=False).create(root/"env")
python=str(root/"env/bin/python")
command=[sys.executable,"-m","pip","--python",python,"install","--no-cache-dir","vllm==0.21.0","transformers==4.57.6"]
log=Path("/kaggle/working/edi-free-vllm-install.log")
print("Installing pinned vLLM in an isolated environment; progress saved to",log,flush=True)
with log.open("w") as out:
    result=subprocess.run(command,stdout=out,stderr=subprocess.STDOUT,timeout=1200)
report={"stage":"isolated_install","exit_code":result.returncode,"free_disk_before":free,"paid_fallback":False,
        "benchmark_started":False,"command":command}
if result.returncode==0:
    check=subprocess.run([python,"-c","import torch,vllm,transformers,json; print(json.dumps(dict(torch=torch.__version__,cuda=torch.version.cuda,vllm=vllm.__version__,transformers=transformers.__version__,cuda_available=torch.cuda.is_available())))"],text=True,capture_output=True,timeout=90)
    report["import_exit_code"]=check.returncode
    report["import_stdout"]=check.stdout[-2000:]
    report["import_stderr"]=check.stderr[-2000:]
    freeze=subprocess.run([sys.executable,"-m","pip","--python",python,"freeze"],text=True,capture_output=True)
    Path("/kaggle/working/edi-free-vllm-packages.txt").write_text(freeze.stdout)
Path("/kaggle/working/edi-free-vllm-install.json").write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2),flush=True)
print("INSTALL LOG TAIL:\n"+"\n".join(log.read_text(errors="replace").splitlines()[-12:]),flush=True)


## Neutral inference


In [ ]:
# Launch the pinned neutral probe once, using the isolated environment.
import json,os,subprocess,time
from pathlib import Path
out=Path("/kaggle/working")
installed=json.loads((out/"edi-free-vllm-install.json").read_text())
assert installed.get("exit_code")==0 and installed.get("import_exit_code")==0, installed
source='"""Two neutral requests to a loopback vLLM server; no EDI benchmark."""\nimport base64, hashlib, io, json, os, secrets, signal, subprocess, sys, time, traceback\nfrom pathlib import Path\nfrom urllib.request import Request, urlopen\nfrom urllib.error import URLError\nfrom huggingface_hub import HfApi, snapshot_download\nfrom PIL import Image, ImageDraw\nROOT=Path("/kaggle/temp/edi-free-vllm")\nOUT=Path("/kaggle/working")\nREPORT={"stage":"starting","sampler":"pytorch","profile_version":"kaggle-t4-fp16-v1","benchmark_started":False,"paid_fallback":False,"model":"Qwen/Qwen3-VL-8B-Instruct",\n        "profile":{"dtype":"float16","tensor_parallel_size":2,"max_model_len":4096,"image_count":1,\n                   "max_image_pixels":65536,"max_output_tokens_per_request":32,"temperature":0,"seed":17}}\ndef update(stage,**fields):\n    REPORT.update(stage=stage,**fields)\n    temp=OUT/"edi-free-vllm-probe.json.tmp"\n    temp.write_text(json.dumps(REPORT,indent=2))\n    temp.replace(OUT/"edi-free-vllm-probe.json")\n    print(stage,flush=True)\ndef post(body,key):\n    request=Request("http://127.0.0.1:8008/v1/chat/completions",data=json.dumps(body).encode(),\n                    headers={"Content-Type":"application/json","Authorization":"Bearer "+key})\n    with urlopen(request,timeout=120) as response:\n        return json.load(response)\nserver=None\nlog=None\ntry:\n    import torch, vllm, transformers\n    update("resolving_model",versions={"torch":torch.__version__,"cuda":torch.version.cuda,\n                                      "vllm":vllm.__version__,"transformers":transformers.__version__})\n    info=HfApi(token=False).model_info(REPORT["model"],revision=\'0c351dd01ed87e9c1b53cbc748cba10e6187ff3b\',files_metadata=True)\n    revision=info.sha\n    if not revision or len(revision)!=40:\n        raise RuntimeError("No exact model revision")\n    expected={f.rfilename:getattr(f.lfs,"sha256",None) for f in info.siblings if getattr(f,"lfs",None)}\n    update("downloading_model",model_revision=revision)\n    snapshot=Path(snapshot_download(REPORT["model"],revision=revision,token=False,\n        cache_dir=str(ROOT/"hf-cache"),\n        allow_patterns=["*.json","*.safetensors","*.txt","*.jinja","*.model"],max_workers=4))\n    manifest=[]\n    update("hashing_model")\n    for file in sorted(snapshot.rglob("*")):\n        if not file.is_file():\n            continue\n        digest=hashlib.file_digest(file.open("rb"),"sha256").hexdigest()\n        relative=file.relative_to(snapshot).as_posix()\n        if expected.get(relative) and expected[relative]!=digest:\n            raise RuntimeError("Model file digest mismatch: "+relative)\n        manifest.append({"path":relative,"bytes":file.stat().st_size,"sha256":digest})\n    (OUT/"edi-free-qwen-artifacts.json").write_text(json.dumps(\n        {"repository":REPORT["model"],"revision":revision,"files":manifest},indent=2))\n    key=secrets.token_urlsafe(32)\n    env=dict(os.environ,HF_HUB_OFFLINE="1",TRANSFORMERS_OFFLINE="1",\n             HF_HUB_DISABLE_IMPLICIT_TOKEN="1",VLLM_NO_USAGE_STATS="1",DO_NOT_TRACK="1",\n             VLLM_API_KEY=key,VLLM_WORKER_MULTIPROC_METHOD="spawn",VLLM_USE_FLASHINFER_SAMPLER="0")\n    command=[sys.executable,"-m","vllm.entrypoints.openai.api_server",\n        "--model",str(snapshot),"--served-model-name",REPORT["model"],\n        "--tokenizer",str(snapshot),"--host","127.0.0.1","--port","8008",\n        "--dtype","half","--tensor-parallel-size","2","--max-model-len","4096",\n        "--gpu-memory-utilization","0.90","--max-num-seqs","1",\n        "--max-num-batched-tokens","4096","--enforce-eager","--disable-custom-all-reduce",\n        "--load-format","safetensors","--attention-backend","TRITON_ATTN",\n        "--mm-encoder-attn-backend","TORCH_SDPA",\n        "--limit-mm-per-prompt",json.dumps({"image":1,"video":0}),\n        "--mm-processor-kwargs",json.dumps({"min_pixels":4096,"max_pixels":65536})]\n    env.pop("PYTHONPATH",None)\n    env.pop("PYTHONHOME",None)\n    log=(OUT/"edi-free-vllm-server.log").open("w")\n    update("starting_server",server_command=command)\n    server=subprocess.Popen(command,stdout=log,stderr=subprocess.STDOUT,env=env,start_new_session=True)\n    deadline=time.monotonic()+720\n    while time.monotonic()<deadline:\n        if server.poll() is not None:\n            raise RuntimeError("vLLM server exited with "+str(server.returncode))\n        try:\n            request=Request("http://127.0.0.1:8008/v1/models",headers={"Authorization":"Bearer "+key})\n            with urlopen(request,timeout=2) as response:\n                catalog=json.load(response)\n            break\n        except (URLError,TimeoutError):\n            time.sleep(2)\n    else:\n        raise TimeoutError("vLLM startup deadline exceeded")\n    update("neutral_text_request")\n    text_response=post(dict(model=REPORT["model"],messages=[{"role":"user","content":"What is 1 + 1? Reply with the digit only."}],\n                            max_tokens=32,temperature=0,seed=17),key)\n    image=Image.new("RGB",(256,128),"white")\n    draw=ImageDraw.Draw(image)\n    draw.rectangle((16,16,112,112),fill=(255,0,0))\n    draw.ellipse((144,16,240,112),fill=(0,0,255))\n    raw=io.BytesIO(); image.save(raw,format="PNG"); data=raw.getvalue()\n    (OUT/"edi-free-neutral-image.png").write_bytes(data)\n    update("neutral_image_request",image_sha256=hashlib.sha256(data).hexdigest())\n    image_response=post(dict(model=REPORT["model"],messages=[{"role":"user","content":[\n        {"type":"text","text":"Name the shape and its color on the left, then on the right."},\n        {"type":"image_url","image_url":{"url":"data:image/png;base64,"+base64.b64encode(data).decode()}}]}],\n        max_tokens=32,temperature=0,seed=17),key)\n    (OUT/"edi-free-neutral-responses.json").write_text(json.dumps(\n        {"text":text_response,"image":image_response},indent=2))\n    update("neutral_requests_completed",\n        text_answer=text_response["choices"][0]["message"]["content"],\n        image_answer=image_response["choices"][0]["message"]["content"],\n        text_usage=text_response.get("usage"),image_usage=image_response.get("usage"))\nexcept Exception as error:\n    update("failed",error_type=type(error).__name__,error=str(error)[:500])\n    traceback.print_exc()\nfinally:\n    if server and server.poll() is None:\n        os.killpg(server.pid,signal.SIGTERM)\n        try:\n            server.wait(timeout=30)\n        except subprocess.TimeoutExpired:\n            os.killpg(server.pid,signal.SIGKILL)\n            server.wait(timeout=10)\n    if log:\n        log.close()\n    REPORT["server_stopped"]=True\n    update(REPORT["stage"])\n'
script=out/"qwen_neutral_probe.py"; script.write_text(source)
env=dict(os.environ,HF_HUB_DISABLE_IMPLICIT_TOKEN="1",HF_HUB_DISABLE_PROGRESS_BARS="1")
env.pop("PYTHONPATH",None); env.pop("PYTHONHOME",None)
with (out/"edi-free-qwen-probe.log").open("w") as log:
    child=subprocess.Popen(["/kaggle/temp/edi-free-vllm/env/bin/python",str(script)],
        stdout=log,stderr=subprocess.STDOUT,env=env,start_new_session=True)
(out/"edi-free-qwen-process.json").write_text(json.dumps({"pid":child.pid,"started_at":time.time()}))
print("Neutral probe PID",child.pid,"started; no EDI benchmark has started.")


## Read-only monitor


In [ ]:
# Read-only monitor for the current neutral probe.
import json,subprocess,time
from pathlib import Path
out=Path("/kaggle/working")
r=json.loads((out/"edi-free-vllm-probe.json").read_text())
print("STATUS:",r.get("stage"),"PROFILE:",r.get("profile_version"),"SAMPLER:",r.get("sampler","default"))
print("REVISION:",r.get("model_revision"))
print("TEXT:",r.get("text_answer"),"IMAGE:",r.get("image_answer"))
print("ERROR:",r.get("error"),"SERVER STOPPED:",r.get("server_stopped"))
print("USAGE:",r.get("text_usage"),r.get("image_usage"))
print(subprocess.run(["nvidia-smi","--query-gpu=memory.used,utilization.gpu","--format=csv,noheader"],text=True,capture_output=True).stdout)
f=out/"edi-free-vllm-server.log"
if f.exists(): print("SERVER LOG TAIL:\n"+"\n".join(f.read_text(errors="replace").splitlines()[-5:]))
